In [1]:
# Import required libraries
import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Load the original Heart Failure dataset
df = pd.read_csv(
    "../datasets/heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

In [ ]:
# Verify the original dataset
print("Dataset Shape:", df.shape)

print(
    "Original Missing Values:",
    df.isnull().sum().sum()
)

## Create Numerical Missing Values

The original dataset contains no missing values.

A separate copy will be created, and missing values will be introduced only for practical learning.

In [ ]:
# Create an independent practice copy
df_missing = df.copy()

# Missing values for Mean Imputation
df_missing.loc[
    [5, 10, 15, 20, 25],
    "age"
] = np.nan

# Missing values for Median Imputation
df_missing.loc[
    [30, 35, 40, 45, 50],
    "serum_creatinine"
] = np.nan

# Missing values for Mode Imputation
df_missing.loc[
    [55, 60, 65, 70],
    "ejection_fraction"
] = np.nan

# Display missing counts
df_missing[
    [
        "age",
        "serum_creatinine",
        "ejection_fraction"
    ]
].isnull().sum()

In [ ]:
# Store indexes for later comparison

age_missing_indexes = (
    df_missing[
        df_missing["age"].isnull()
    ].index
)

creatinine_missing_indexes = (
    df_missing[
        df_missing["serum_creatinine"].isnull()
    ].index
)

ejection_missing_indexes = (
    df_missing[
        df_missing["ejection_fraction"].isnull()
    ].index
)

print("Age Missing Indexes:", age_missing_indexes.tolist())

print(
    "Serum Creatinine Missing Indexes:",
    creatinine_missing_indexes.tolist()
)

print(
    "Ejection Fraction Missing Indexes:",
    ejection_missing_indexes.tolist()
)

# Mean Imputation

Mean Imputation replaces missing values with the average of the available values.

It is most suitable for approximately symmetric numerical data without strong outliers.

In [ ]:
# Calculate the mean of Age
age_mean = df_missing["age"].mean()

print(f"Age Mean: {age_mean:.2f}")

In [ ]:
# Create a separate copy for Mean Imputation
df_mean = df_missing.copy()

# Fill missing Age values with the mean
df_mean["age"] = df_mean["age"].fillna(
    age_mean
)

# Display imputed rows
df_mean.loc[
    age_missing_indexes,
    ["age"]
]

In [ ]:
print(
    "Remaining Missing Age Values:",
    df_mean["age"].isnull().sum()
)

# Median Imputation

Median Imputation replaces missing values with the middle value of the feature.

It is suitable for skewed data and features containing outliers.

In [ ]:
# Calculate median Serum Creatinine
creatinine_median = (
    df_missing["serum_creatinine"]
    .median()
)

print(
    "Serum Creatinine Median:",
    creatinine_median
)

In [ ]:
# Create a separate copy for Median Imputation
df_median = df_missing.copy()

df_median["serum_creatinine"] = (
    df_median["serum_creatinine"]
    .fillna(creatinine_median)
)

df_median.loc[
    creatinine_missing_indexes,
    ["serum_creatinine"]
]

In [ ]:
print(
    "Remaining Missing Serum Creatinine Values:",
    df_median["serum_creatinine"]
    .isnull()
    .sum()
)

# Mode Imputation

Mode Imputation replaces missing values with the most frequently occurring value.

It is useful for discrete numerical variables.

In [ ]:
# Calculate the most frequent Ejection Fraction value
ejection_mode = (
    df_missing["ejection_fraction"]
    .mode()[0]
)

print(
    "Ejection Fraction Mode:",
    ejection_mode
)

In [ ]:
# Create a separate copy for Mode Imputation
df_mode = df_missing.copy()

df_mode["ejection_fraction"] = (
    df_mode["ejection_fraction"]
    .fillna(ejection_mode)
)

df_mode.loc[
    ejection_missing_indexes,
    ["ejection_fraction"]
]

In [ ]:
print(
    "Remaining Missing Ejection Fraction Values:",
    df_mode["ejection_fraction"]
    .isnull()
    .sum()
)

# Conditional Mean Imputation

Conditional Mean Imputation fills missing values using the mean of a meaningful group.

For practice, missing Age values will be filled using the mean Age within each `diabetes` group.

In [ ]:
# Create a separate copy
df_conditional = df_missing.copy()

# Fill missing Age values using the
# mean Age of each diabetes group
df_conditional["age"] = (
    df_conditional
    .groupby("diabetes")["age"]
    .transform(
        lambda group: group.fillna(
            group.mean()
        )
    )
)

df_conditional.loc[
    age_missing_indexes,
    ["diabetes", "age"]
]

In [ ]:
# Display Age mean for each diabetes group
group_age_means = (
    df_missing
    .groupby("diabetes")["age"]
    .mean()
)

group_age_means

In [ ]:
print(
    "Remaining Missing Age Values:",
    df_conditional["age"]
    .isnull()
    .sum()
)

# KNN Imputation

KNN Imputation estimates missing values using similar observations.

For demonstration, selected continuous numerical features will be used.

> KNN is distance-based and can be affected by different feature scales. Scaling will be covered in its dedicated topic.

In [ ]:
# Select continuous numerical features
knn_features = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium"
]

df_knn = df_missing.copy()

df_knn[knn_features].isnull().sum()

In [ ]:
# Create KNN imputer
knn_imputer = KNNImputer(
    n_neighbors=5
)

# Fit and transform selected features
knn_imputed_array = knn_imputer.fit_transform(
    df_knn[knn_features]
)

# Convert result back into a DataFrame
df_knn_imputed = pd.DataFrame(
    knn_imputed_array,
    columns=knn_features,
    index=df_knn.index
)

df_knn_imputed.head()

In [ ]:
# Replace numerical columns with imputed values
df_knn[knn_features] = df_knn_imputed

print(
    "Remaining Missing Values in KNN Features:",
    df_knn[knn_features]
    .isnull()
    .sum()
    .sum()
)

In [ ]:
# Display selected rows that originally contained missing values
all_missing_indexes = sorted(
    set(age_missing_indexes)
    | set(creatinine_missing_indexes)
    | set(ejection_missing_indexes)
)

df_knn.loc[
    all_missing_indexes,
    [
        "age",
        "ejection_fraction",
        "serum_creatinine"
    ]
]

# Regression Imputation

Regression Imputation trains a model using complete rows and predicts the missing values.

In this example:

- Target feature: `serum_creatinine`
- Predictor features:
  - `age`
  - `ejection_fraction`
  - `platelets`
  - `serum_sodium`

In [ ]:
# Create an independent copy
df_regression = df_missing.copy()

target_column = "serum_creatinine"

predictor_columns = [
    "age",
    "ejection_fraction",
    "platelets",
    "serum_sodium"
]

# Use only rows where predictors are complete
predictors_complete = (
    df_regression[predictor_columns]
    .notnull()
    .all(axis=1)
)

# Complete target rows for training
training_mask = (
    df_regression[target_column].notnull()
    & predictors_complete
)

# Missing target rows for prediction
prediction_mask = (
    df_regression[target_column].isnull()
    & predictors_complete
)

print(
    "Training Rows:",
    training_mask.sum()
)

print(
    "Rows to Predict:",
    prediction_mask.sum()
)

In [ ]:
# Prepare training data
X_train = df_regression.loc[
    training_mask,
    predictor_columns
]

y_train = df_regression.loc[
    training_mask,
    target_column
]

# Create and train model
regression_model = LinearRegression()

regression_model.fit(
    X_train,
    y_train
)

print("Regression Model Trained")

In [ ]:
# Prepare rows with missing target values
X_missing = df_regression.loc[
    prediction_mask,
    predictor_columns
]

# Predict missing Serum Creatinine values
predicted_values = regression_model.predict(
    X_missing
)

predicted_values

In [ ]:
# Fill missing target values with predictions
df_regression.loc[
    prediction_mask,
    target_column
] = predicted_values

# Display imputed rows
df_regression.loc[
    prediction_mask,
    predictor_columns + [target_column]
]

In [ ]:
print(
    "Remaining Missing Serum Creatinine Values:",
    df_regression[target_column]
    .isnull()
    .sum()
)

# Compare Imputed Values

The following comparison shows how different methods can generate different replacement values for the same missing records.

In [ ]:
# Compare Mean, Conditional Mean and KNN for missing Age
age_comparison = pd.DataFrame({
    "Original Missing": (
        df_missing.loc[
            age_missing_indexes,
            "age"
        ]
    ),
    "Mean Imputation": (
        df_mean.loc[
            age_missing_indexes,
            "age"
        ]
    ),
    "Conditional Mean": (
        df_conditional.loc[
            age_missing_indexes,
            "age"
        ]
    ),
    "KNN Imputation": (
        df_knn.loc[
            age_missing_indexes,
            "age"
        ]
    )
})

age_comparison

In [ ]:
# Compare Median, KNN and Regression methods
creatinine_comparison = pd.DataFrame({
    "Original Missing": (
        df_missing.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    ),
    "Median Imputation": (
        df_median.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    ),
    "KNN Imputation": (
        df_knn.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    ),
    "Regression Imputation": (
        df_regression.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    )
})

creatinine_comparison

In [ ]:
# Verify the selected imputed copies

verification = pd.DataFrame({
    "Method": [
        "Mean",
        "Median",
        "Mode",
        "Conditional Mean",
        "KNN",
        "Regression"
    ],
    "Relevant Missing Values": [
        df_mean["age"].isnull().sum(),
        df_median["serum_creatinine"].isnull().sum(),
        df_mode["ejection_fraction"].isnull().sum(),
        df_conditional["age"].isnull().sum(),
        df_knn[knn_features].isnull().sum().sum(),
        df_regression["serum_creatinine"].isnull().sum()
    ]
})

verification

In [ ]:
print("Original Dataset Shape:", df.shape)

print(
    "Original Dataset Missing Values:",
    df.isnull().sum().sum()
)

# Summary

In this notebook, we:

- Preserved the original Heart Failure dataset
- Introduced numerical missing values in a practice copy
- Applied Mean Imputation
- Applied Median Imputation
- Applied Mode Imputation
- Applied Conditional Mean Imputation
- Applied KNN Imputation
- Applied Regression Imputation
- Compared replacement values generated by different techniques
- Verified that the original dataset remained unchanged

## Key Learnings

- Mean is suitable for approximately symmetric data.
- Median is more robust for skewed data and outliers.
- Mode is suitable for discrete numerical features.
- Conditional Mean preserves group-level differences.
- KNN uses similar observations.
- Regression predicts missing values using other features.
- Different methods produce different estimates.
- Imputation should be selected through analysis and validation.

## Next Topic

Categorical Missing Value Imputation